#Department Table Cleaning

##importing dependency and data

In [0]:
import pyspark.sql.functions as F

In [0]:
df_employee_raw = spark.read.table("bronze.azure_blob_storage.employee")

In [0]:
df_employee_raw.display()

##Removing columns inserted by fivetron

In [0]:
df_employee=df_employee_raw.drop('_file','_line','_modified','_fivetran_synced')

In [0]:
df_employee.display()

## Type Casting

#### converting to int

In [0]:
df_employee=df_employee.withColumn("employee_id",F.col("employee_id").cast("int"))
df_employee=df_employee.withColumn("department_id",F.col("department_id").cast("int"))
df_employee=df_employee.withColumn("company_id",F.col("company_id").cast("int"))
df_employee.display()

#### Date and time handling

In [0]:
df_employee = df_employee.withColumn("hire_date", F.to_date(F.col("hire_date"), "dd-MM-yyyy HH:mm"))

In [0]:
df_employee.display()

In [0]:
df_employee = df_employee.withColumn(
    "termination_date",
    F.to_date(F.col("termination_date"), "dd-MM-yyyy HH:mm")
)
df_employee.display()

## Handling null values

In [0]:
for column in df_employee.columns:
    null_count = df_employee.filter(F.col(column).isNull()).count()
    print(f"{column}: {null_count}")

In [0]:
df_employee = df_employee.withColumn(
    "termination_date",
    F.when(
        F.col("termination_date").isNull(),
        F.date_format(F.current_date(), "yyyy-MM-dd")
    ).otherwise(F.date_format(F.col("termination_date"), "yyyy-MM-dd"))
)
df_employee.display()

## Handling Duplicates

In [0]:
if df_employee.count() > df_employee.dropDuplicates().count():
    df_employee = df_employee.dropDuplicates()
df_employee.display()

In [0]:
df_employee.display()

## Writing to Silver

In [0]:
df_employee.write.mode("overwrite").saveAsTable("silver.transformation.employee")